# 02a — Agent Basics
**Block B · SAPA AI Workshop**

In this notebook you will build the simplest possible agent: one LLM + one tool + one persona.
This is the unit from which everything else is composed.

> **Before you start:** make sure your `ANTHROPIC_API_KEY` is set.
> Run `echo $ANTHROPIC_API_KEY` in the terminal to verify.

---
## Step 1 — Load your API key

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set — check your .env or Codespaces secret"
print("API key loaded.")

---
## Step 2 — Define a Tool

A tool is a plain Python function decorated with `@tool`.

**Two things the LLM reads:**
- The **docstring** — tells the LLM *when* to call this tool and *what it does*
- The **type hints** — tells the LLM *how* to format the arguments

Write these clearly and the LLM will use the tool correctly.

In [ ]:
from langchain_core.tools import tool

@tool
def calculate_severity_score(event_description: str) -> dict:
    """Calculate a numerical severity score (1-10) for an adverse event description.

    Args:
        event_description: Free-text description of the adverse event or outcome.
    """
    severity_map = {
        "death": 10, "fatal": 10,
        "life-threatening": 9,
        "hospitalized": 7, "serious": 7, "severe": 7,
        "moderate": 5,
        "mild": 2, "minor": 2,
    }
    text = event_description.lower()
    best_score, best_term = 1, "unspecified"
    for term, score in severity_map.items():
        if term in text and score > best_score:
            best_score, best_term = score, term
    return {"score": best_score, "matched_term": best_term, "scale": "1-10"}


# Try calling it directly first — it's just a Python function
result = calculate_severity_score.invoke({"event_description": "Patient was hospitalized with severe nausea"})
print("Direct tool call result:", result)

---
## Step 3 — Create an Agent with a Persona and a Tool

The **persona** is the system prompt — it tells the LLM what role it plays.
The **tool** extends what the agent can do beyond text generation.

`create_react_agent` wires the LLM + tools + the ReAct loop for you.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langgraph.prebuilt import create_react_agent

# The LLM — the reasoning engine
llm = ChatAnthropic(model="claude-haiku-4-5-20251001")

# The persona — scopes the agent's role and behavior
SYSTEM_PROMPT = """You are a pharmacovigilance specialist.
When asked to assess an adverse event, use the calculate_severity_score tool
and include the score in your response. Be concise."""

# Assemble the agent
agent = create_react_agent(
    llm,
    tools=[calculate_severity_score],
    prompt=SYSTEM_PROMPT,
)

print("Agent created.")

---
## Step 4 — Run the Agent

Watch the output carefully. You should see the agent:
1. **Reason** — decide to call the severity tool
2. **Act** — call `calculate_severity_score`
3. **Observe** — receive the score
4. **Respond** — write its final assessment using the score

This is the ReAct loop in action.

In [ ]:
SAMPLE_REPORT = """
Patient: 45-year-old female.
Drug: Metformin 500mg twice daily.
Onset: 2024-01-15.
Adverse event: Severe nausea, vomiting, and lactic acidosis.
Outcome: Hospitalized for 3 days.
Reporter: Prescribing physician.
"""

result = agent.invoke({
    "messages": [{"role": "user", "content": f"Assess this adverse event report:\n{SAMPLE_REPORT}"}]
})

print("=" * 60)
print("AGENT RESPONSE")
print("=" * 60)
print(result["messages"][-1].content)

---
## What Just Happened?

You built an agent with:
- **LLM**: Claude Haiku — the reasoning engine
- **Tool**: `calculate_severity_score` — a concrete action it can take
- **Persona**: a pharmacovigilance specialist — the role that shapes its response
- **Memory**: the message history passed through the ReAct loop

This single agent is the building block. In `02b_orchestration.ipynb` you will see how **multiple agents like this one** are composed into a coordinated pipeline using LangGraph.

> **Next:** open `notebooks/02b_orchestration.ipynb`